In [ ]:
# Supervised Models – Part 2
# This notebook contains:
# - Decision Tree
# - Random Forest
# - Gradient Boosting (Ensemble)

In [4]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/bharatgarg1234/diabetes/diabetes_binary_5050split_health_indicators_BRFSS2015.csv")
df.head()

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,3.0,5.0,30.0,0.0,1.0,4.0,6.0,8.0
1,0.0,1.0,1.0,1.0,26.0,1.0,1.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,12.0,6.0,8.0
2,0.0,0.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,10.0,0.0,1.0,13.0,6.0,8.0
3,0.0,1.0,1.0,1.0,28.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,3.0,0.0,3.0,0.0,1.0,11.0,6.0,8.0
4,0.0,0.0,0.0,1.0,29.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,8.0,5.0,8.0


In [5]:
# ============================================================
# Compare DT & RF vs SVM & LR using 10-fold CV
# Evaluate: accuracy, precision, recall, F1
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

RANDOM_SEED = 42

# Target column
target_col = "Diabetes_binary"
assert target_col in df.columns, f"Target column '{target_col}' not found."

X = df.drop(columns=[target_col, "GenHlth"])
y = df[target_col]

print("Class distribution:\n", y.value_counts(normalize=True).round(3))

# ---------------- Train/test split (80/20) ----------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_SEED
)

# ---------------- CV setup ----------------
k = 10
cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_SEED)

scoring = {
    "accuracy": "accuracy",
    "precision_weighted": "precision_weighted",
    "recall_weighted": "recall_weighted",
    "f1_weighted": "f1_weighted",
}

# ---------------- Helper functions ----------------
def summarize_cv(name, cv_res):
    """Return mean/std summary for cross_validate results"""
    return {
        "model": name,
        "acc_mean": np.mean(cv_res["test_accuracy"]),
        "acc_std": np.std(cv_res["test_accuracy"]),
        "prec_mean": np.mean(cv_res["test_precision_weighted"]),
        "prec_std": np.std(cv_res["test_precision_weighted"]),
        "rec_mean": np.mean(cv_res["test_recall_weighted"]),
        "rec_std": np.std(cv_res["test_recall_weighted"]),
        "f1_mean": np.mean(cv_res["test_f1_weighted"]),
        "f1_std": np.std(cv_res["test_f1_weighted"]),
    }

def test_metrics(name, clf, X_tr, y_tr, X_te, y_te):
    """Fit on train, evaluate on test set"""
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    return {
        "model": name,
        "acc": accuracy_score(y_te, y_pred),
        "prec": precision_score(y_te, y_pred, average="weighted", zero_division=0),
        "rec": recall_score(y_te, y_pred, average="weighted", zero_division=0),
        "f1": f1_score(y_te, y_pred, average="weighted", zero_division=0),
    }

# ============================================================
# 1) DT & RF: 10-fold CV on TRAINING SET
# ============================================================
cv_rows = []

# Decision Tree (CV only)
dt = DecisionTreeClassifier(random_state=RANDOM_SEED)
dt_cv = cross_validate(dt, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
cv_rows.append(summarize_cv("DT (CV)", dt_cv))

# Random Forest with varying #trees
rf_tree_counts = [50, 100, 150, 200]
rf_cv_summaries = {}
for n in rf_tree_counts:
    rf = RandomForestClassifier(n_estimators=n, random_state=RANDOM_SEED, n_jobs=-1)
    rf_cv = cross_validate(rf, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    row = summarize_cv(f"RF_{n}t (CV)", rf_cv)
    cv_rows.append(row)
    rf_cv_summaries[n] = row

cv_table = pd.DataFrame(cv_rows).sort_values(by="f1_mean", ascending=False).reset_index(drop=True)
print("\n=== 10-fold CV on TRAIN set (DT & RF) ===")
print(cv_table.round(4))

# Optional: variance in RF F1 across number of trees
rf_var_table = pd.DataFrame.from_dict(rf_cv_summaries, orient="index")[
    ["f1_mean", "f1_std", "acc_mean", "acc_std"]
].rename_axis("n_trees").sort_index()
print("\n--- RF performance vs number of trees (CV, train set) ---")
print(rf_var_table.round(4))

# ============================================================
# 2) Compare DT, RF, SVM, LR on held-out TEST SET
# ============================================================
test_rows = []

# Decision Tree
test_rows.append(test_metrics("DT (test)", DecisionTreeClassifier(random_state=RANDOM_SEED),
                              X_train, y_train, X_test, y_test))

# Random Forests
for n in rf_tree_counts:
    test_rows.append(test_metrics(f"RF_{n}t (test)",
                                  RandomForestClassifier(n_estimators=n, random_state=RANDOM_SEED, n_jobs=-1),
                                  X_train, y_train, X_test, y_test))

# SVM (RBF)
# test_rows.append(test_metrics("SVM_rbf (test)",
#                               SVC(kernel="rbf", C=1.0, gamma="scale", random_state=RANDOM_SEED),
#                               X_train, y_train, X_test, y_test))

# Logistic Regression
# test_rows.append(test_metrics("LR (test)",
#                               LogisticRegression(max_iter=1000, n_jobs=-1, random_state=RANDOM_SEED),
#                               X_train, y_train, X_test, y_test))

test_table = pd.DataFrame(test_rows).sort_values(by="f1", ascending=False).reset_index(drop=True)
print("\n=== Held-out TEST performance (all models) ===")
print(test_table.round(4))


Class distribution:
 Diabetes_binary
0.0    0.5
1.0    0.5
Name: proportion, dtype: float64

=== 10-fold CV on TRAIN set (DT & RF) ===
          model  acc_mean  acc_std  prec_mean  prec_std  rec_mean  rec_std  \
0  RF_200t (CV)    0.7215   0.0033     0.7225    0.0035    0.7215   0.0033   
1  RF_150t (CV)    0.7205   0.0036     0.7215    0.0037    0.7205   0.0036   
2  RF_100t (CV)    0.7197   0.0042     0.7206    0.0042    0.7197   0.0042   
3   RF_50t (CV)    0.7186   0.0040     0.7192    0.0040    0.7186   0.0040   
4       DT (CV)    0.6416   0.0047     0.6417    0.0047    0.6416   0.0047   

   f1_mean  f1_std  
0   0.7211  0.0033  
1   0.7202  0.0036  
2   0.7194  0.0042  
3   0.7184  0.0040  
4   0.6416  0.0047  

--- RF performance vs number of trees (CV, train set) ---
         f1_mean  f1_std  acc_mean  acc_std
n_trees                                    
50        0.7184  0.0040    0.7186   0.0040
100       0.7194  0.0042    0.7197   0.0042
150       0.7202  0.0036    0.7205 

In [8]:
# ============================================================
# Decision Tree — Test Set Performance (80/20 Split)
# ============================================================
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
dt = DecisionTreeClassifier(random_state=RANDOM_SEED)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

dt_acc  = accuracy_score(y_test, dt_pred)
dt_prec = precision_score(y_test, dt_pred, average="weighted", zero_division=0)
dt_rec  = recall_score(y_test, dt_pred, average="weighted", zero_division=0)
dt_f1   = f1_score(y_test, dt_pred, average="weighted", zero_division=0)

print("\n=== Decision Tree — Test Set Performance (80/20) ===")
print(f"Accuracy:  {dt_acc:.4f}")
print(f"Precision: {dt_prec:.4f}")
print(f"Recall:    {dt_rec:.4f}")
print(f"F1-Score:  {dt_f1:.4f}")
# ============================================================
# Decision Tree — Confusion Matrix (80/20 Split)
# ============================================================

dt_cm = confusion_matrix(y_test, dt_pred)
print("\n=== Decision Tree — Confusion Matrix (80/20) ===")
print(dt_cm)


=== Decision Tree — Test Set Performance (80/20) ===
Accuracy:  0.6434
Precision: 0.6435
Recall:    0.6434
F1-Score:  0.6434

=== Decision Tree — Confusion Matrix (80/20) ===
[[4625 2445]
 [2597 4472]]


In [12]:
# ============================================================
# Random Forest — Train/Test Split Experiments
# ============================================================

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

splits = [0.2, 0.3, 0.4]
print("=== Train/Test Split Experiments ===\n")

for split in splits:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=split, stratify=y, random_state=42
    )

    rf = RandomForestClassifier(random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted")
    rec = recall_score(y_test, y_pred, average="weighted")
    f1 = f1_score(y_test, y_pred, average="weighted")
    cm = confusion_matrix(y_test, y_pred)

    print(f"Split: {split}")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1 Score:  {f1:.4f}")
    print(f"  Confusion Matrix:\n{cm}\n")


=== Train/Test Split Experiments ===

Split: 0.2
  Accuracy:  0.7192
  Precision: 0.7201
  Recall:    0.7192
  F1 Score:  0.7189
  Confusion Matrix:
[[4855 2215]
 [1755 5314]]

Split: 0.3
  Accuracy:  0.7184
  Precision: 0.7192
  Recall:    0.7184
  F1 Score:  0.7181
  Confusion Matrix:
[[7283 3321]
 [2652 7952]]

Split: 0.4
  Accuracy:  0.7203
  Precision: 0.7213
  Recall:    0.7203
  F1 Score:  0.7200
  Confusion Matrix:
[[ 9727  4412]
 [ 3496 10642]]



In [4]:
# ============================================================
# Random Forest — K-Fold Experiments
# ============================================================

print("=== K-Fold CV Experiments ===\n")

k_values = [3, 5, 10]

for k in k_values:
    cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

    rf = RandomForestClassifier(random_state=42, n_jobs=-1)
    cv_res = cross_validate(
        rf, X, y, cv=cv,
        scoring=["accuracy", "precision_weighted", "recall_weighted", "f1_weighted"],
        n_jobs=-1
    )

    print(f"K-Fold: {k}")
    print(f"  CV Accuracy Mean:  {cv_res['test_accuracy'].mean():.4f}")
    print(f"  CV Precision Mean: {cv_res['test_precision_weighted'].mean():.4f}")
    print(f"  CV Recall Mean:    {cv_res['test_recall_weighted'].mean():.4f}")
    print(f"  CV F1 Mean:        {cv_res['test_f1_weighted'].mean():.4f}")
    print(f"  CV F1 Std:         {cv_res['test_f1_weighted'].std():.4f}\n")
    print(f"  CV Accuracy Std:  {cv_res['test_accuracy'].std():.4f}")


=== K-Fold CV Experiments ===

K-Fold: 3
  CV Accuracy Mean:  0.7208
  CV Precision Mean: 0.7218
  CV Recall Mean:    0.7208
  CV F1 Mean:        0.7205
  CV F1 Std:         0.0016

  CV Accuracy Std:  0.0015
K-Fold: 5
  CV Accuracy Mean:  0.7198
  CV Precision Mean: 0.7207
  CV Recall Mean:    0.7198
  CV F1 Mean:        0.7195
  CV F1 Std:         0.0026

  CV Accuracy Std:  0.0027
K-Fold: 10
  CV Accuracy Mean:  0.7200
  CV Precision Mean: 0.7210
  CV Recall Mean:    0.7200
  CV F1 Mean:        0.7197
  CV F1 Std:         0.0046

  CV Accuracy Std:  0.0045


In [14]:
# ============================================================
# Random Forest — Random Seed Experiments
# ============================================================

print("=== Random Seed Experiments ===\n")

seeds = [42, 7]

for seed in seeds:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed
    )

    rf = RandomForestClassifier(random_state=seed, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted")
    rec = recall_score(y_test, y_pred, average="weighted")
    f1 = f1_score(y_test, y_pred, average="weighted")
    cm = confusion_matrix(y_test, y_pred)

    print(f"Seed: {seed}")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1 Score:  {f1:.4f}")
    print(f"  Confusion Matrix:\n{cm}\n")


=== Random Seed Experiments ===

Seed: 42
  Accuracy:  0.7192
  Precision: 0.7201
  Recall:    0.7192
  F1 Score:  0.7189
  Confusion Matrix:
[[4855 2215]
 [1755 5314]]

Seed: 7
  Accuracy:  0.7208
  Precision: 0.7216
  Recall:    0.7208
  F1 Score:  0.7206
  Confusion Matrix:
[[4885 2184]
 [1763 5307]]



In [15]:
# ============================================================
# Random Forest — Baseline Model (80/20, seed=42)
# ============================================================

print("=== Baseline RF (80/20, seed=42) ===\n")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

rf = RandomForestClassifier(random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred, average='weighted'):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")


=== Baseline RF (80/20, seed=42) ===

Accuracy:  0.7192
Precision: 0.7201
Recall:    0.7192
F1 Score:  0.7189
Confusion Matrix:
[[4855 2215]
 [1755 5314]]


In [5]:
# ============================================================
# Random Forest — Hyperparameter Tuning
# ============================================================
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import GridSearchCV
print("=== Tuned Random Forest ===\n")

param_grid = {
    "n_estimators": [100, 150, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=5,
    scoring="f1_weighted",
    n_jobs=-1
)

grid.fit(X_train, y_train)
best_rf = grid.best_estimator_

y_pred = best_rf.predict(X_test)

print("Best Parameters:", grid.best_params_)
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred, average='weighted'):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred, average='weighted'):.4f}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")


=== Tuned Random Forest ===

Best Parameters: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 150}
Accuracy:  0.7400
Precision: 0.7417
Recall:    0.7400
F1 Score:  0.7395
Confusion Matrix:
[[4934 2136]
 [1540 5529]]


In [10]:
# =============================================================
# IMPORT LIBRARIES
# =============================================================
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans, BisectingKMeans, DBSCAN
from sklearn.metrics import silhouette_score, rand_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler


# =============================================================
# LOAD AND PREPARE DATA
# =============================================================
df = pd.read_csv("/kaggle/input/datasets/bharatgarg1234/diabetes/diabetes_binary_5050split_health_indicators_BRFSS2015.csv")
# Features and labels
X = df.drop(columns=["Diabetes_binary"]).values
y = df["Diabetes_binary"].values

# Standardize features (IMPORTANT for clustering)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Number of clusters (from true labels)
k = len(np.unique(y))


# =============================================================
# CLASSICAL K-MEANS (Random Init)
# =============================================================
kmeans_random = KMeans(
    n_clusters=k,
    init="random",
    n_init=10,
    random_state=42
)

labels_random = kmeans_random.fit_predict(X_scaled)

sil_random = silhouette_score(X_scaled, labels_random)
rand_random = rand_score(y, labels_random)
ari_random = adjusted_rand_score(y, labels_random)


# =============================================================
# K-MEANS++
# =============================================================
# kmeans_pp = KMeans(
#     n_clusters=k,
#     init="k-means++",
#     n_init=10,
#     random_state=42
# )

# labels_pp = kmeans_pp.fit_predict(X_scaled)

# sil_pp = silhouette_score(X_scaled, labels_pp)
# rand_pp = rand_score(y, labels_pp)


# =============================================================
# BISECTING K-MEANS
# =============================================================
# bisect_kmeans = BisectingKMeans(
#     n_clusters=k,
#     random_state=42
# )

# labels_bisect = bisect_kmeans.fit_predict(X_scaled)

# sil_bisect = silhouette_score(X_scaled, labels_bisect)
# rand_bisect = rand_score(y, labels_bisect)


# =============================================================
# DBSCAN
# =============================================================
# dbscan = DBSCAN(
#     eps=0.5,        # you can tune this
#     min_samples=5
# )

# labels_db = dbscan.fit_predict(X_scaled)

# # Handle noise points (-1)
# mask = labels_db != -1

# # Silhouette only works if >1 cluster
# if len(set(labels_db[mask])) > 1:
#     sil_db = silhouette_score(X_scaled[mask], labels_db[mask])
# else:
#     sil_db = -1  # invalid case

# rand_db = rand_score(y, labels_db)


# =============================================================
# PRINT RESULTS
# =============================================================
print("Clustering Performance:\n")

print("Classical K-Means (Random Init)")
print("Silhouette:", sil_random)
print("Rand Index:", rand_random)
print("ARI:", ari_random)
print()

# print("K-Means++")
# print("Silhouette:", sil_pp)
# print("Rand Index:", rand_pp)
# print()

# print("Bisecting K-Means")
# print("Silhouette:", sil_bisect)
# print("Rand Index:", rand_bisect)
# print()

# print("DBSCAN")
# print("Silhouette:", sil_db)
# print("Rand Index:", rand_db)
# print()


Clustering Performance:

Classical K-Means (Random Init)
Silhouette: 0.16335016983976272
Rand Index: 0.556477322156718
ARI: 0.11295602753290696



In [11]:
# =============================================================
# K-MEANS FOR MULTIPLE k VALUES (Corrected)
# =============================================================

k_values = [2, 3, 4, 5]
results_k = []

for k in k_values:
    kmeans_random = KMeans(
        n_clusters=k,
        init="random",
        n_init=10,
        random_state=42
    )
    
    labels_k = kmeans_random.fit_predict(X_scaled)   
    
    sil = silhouette_score(X_scaled, labels_k)
    ari = adjusted_rand_score(y, labels_k)           
    
    results_k.append([k, sil, ari])

df_kmeans_results = pd.DataFrame(results_k, columns=["k", "Silhouette", "ARI"])

print("K-Means Results for Different k Values:\n")
print(df_kmeans_results)


K-Means Results for Different k Values:

   k  Silhouette       ARI
0  2    0.163350  0.112956
1  3    0.075409  0.149577
2  4    0.082853  0.136574
3  5    0.090381  0.123380


In [5]:
import pandas as pd
import numpy as np

# Your results
results = pd.DataFrame({
    "Model": ["KMeans_Random", "KMeans_PP", "Bisecting_KMeans", "DBSCAN"],
    "Silhouette": [0.16335, 0.16332, 0.16394, 0.16399],
    "RandIndex": [0.55648, 0.55651, 0.55607, 0.50543]
})

# -----------------------------
# 1. WIN COUNT METHOD
# -----------------------------
win_count = {"Model": [], "Wins": []}

for metric in ["Silhouette", "RandIndex"]:
    best_value = results[metric].max()
    best_model = results.loc[results[metric].idxmax(), "Model"]
    
    # Add win to that model
    win_count["Model"].append(best_model)
    win_count["Wins"].append(metric)

# Convert to table
win_df = pd.DataFrame(win_count)
win_summary = win_df["Model"].value_counts().reset_index()
win_summary.columns = ["Model", "WinCount"]

# -----------------------------
# 2. RELATIVE PERFORMANCE INDEX (RPI)
# -----------------------------
def compute_rpi(column):
    worst = results[column].min()
    return results[column] - worst   # worst gets 0

results["RPI_Silhouette"] = compute_rpi("Silhouette")
results["RPI_Rand"] = compute_rpi("RandIndex")
results["RPI_Total"] = results["RPI_Silhouette"] + results["RPI_Rand"]

# -----------------------------
# FINAL OUTPUT
# -----------------------------
print("=== WIN COUNT ===")
print(win_summary, "\n")

print("=== RPI TABLE ===")
print(results[["Model", "RPI_Silhouette", "RPI_Rand", "RPI_Total"]])


=== WIN COUNT ===
       Model  WinCount
0     DBSCAN         1
1  KMeans_PP         1 

=== RPI TABLE ===
              Model  RPI_Silhouette  RPI_Rand  RPI_Total
0     KMeans_Random         0.00003   0.05105    0.05108
1         KMeans_PP         0.00000   0.05108    0.05108
2  Bisecting_KMeans         0.00062   0.05064    0.05126
3            DBSCAN         0.00067   0.00000    0.00067


In [2]:
# =============================================================
# WEEK 7 TUTORIAL (ENSEMBLE)
# =============================================================
# =============================================================
# IMPORT LIBRARIES
# =============================================================
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import StackingClassifier


# =============================================================
# LOAD DATA
# =============================================================
df = pd.read_csv("/kaggle/input/datasets/bharatgarg1234/diabetes/diabetes_binary_5050split_health_indicators_BRFSS2015.csv")

X = df.drop("Diabetes_binary", axis=1)
y = df["Diabetes_binary"]


# =============================================================
# TRAIN–TEST SPLIT
# =============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# =============================================================
# FUNCTION TO EVALUATE MODELS
# =============================================================
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred)
    ]


results = []   # store all model results


# =============================================================
# 1. BOOSTING MODELS
# =============================================================

# ---- AdaBoost ----
ada = AdaBoostClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)
ada.fit(X_train, y_train)
results.append(["AdaBoost"] + evaluate(ada, X_test, y_test))


# ---- Gradient Boosting ----
gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb.fit(X_train, y_train)
results.append(["GradientBoosting"] + evaluate(gb, X_test, y_test))


# ---- XGBoost ----
xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)
xgb.fit(X_train, y_train)
results.append(["XGBoost"] + evaluate(xgb, X_test, y_test))


# =============================================================
# 2. STACKING MODELS
# =============================================================

# ---- Variant A: Base = LR + RF, Meta = DT ----
stack_A = StackingClassifier(
    estimators=[
        ("lr", LogisticRegression(max_iter=1000)),
        ("rf", RandomForestClassifier(n_estimators=100, random_state=42))
    ],
    final_estimator=DecisionTreeClassifier(max_depth=3, random_state=42),
    cv=5,
    stack_method="predict_proba"
)
stack_A.fit(X_train, y_train)
results.append(["Stacking_A (LR+RF → DT)"] + evaluate(stack_A, X_test, y_test))


# ---- Variant B: Base = DT + RF, Meta = LR ----
stack_B = StackingClassifier(
    estimators=[
        ("dt", DecisionTreeClassifier(max_depth=None, random_state=42)),
        ("rf", RandomForestClassifier(n_estimators=100, random_state=42))
    ],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    stack_method="predict_proba"
)
stack_B.fit(X_train, y_train)
results.append(["Stacking_B (DT+RF → LR)"] + evaluate(stack_B, X_test, y_test))


# =============================================================
# 3. RESULTS TABLE
# =============================================================
df_results = pd.DataFrame(
    results,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1"]
)

print("\n=== Ensemble Model Performance Comparison ===\n")
print(df_results)



=== Ensemble Model Performance Comparison ===

                     Model  Accuracy  Precision    Recall        F1
0                 AdaBoost  0.735413   0.718545  0.773943  0.745216
1         GradientBoosting  0.753943   0.733663  0.797284  0.764152
2                  XGBoost  0.753731   0.733013  0.798133  0.764188
3  Stacking_A (LR+RF → DT)  0.746517   0.714567  0.820908  0.764055
4  Stacking_B (DT+RF → LR)  0.732937   0.719680  0.763050  0.740731


In [3]:
# =============================================================
# GRADIENT BOOSTING CONFUSION MATRIX 
# =============================================================
from sklearn.metrics import confusion_matrix

gb_pred = gb.predict(X_test)
cm_gb = confusion_matrix(y_test, gb_pred)

cm_gb


array([[5024, 2046],
       [1433, 5636]])